# 🎤 Job Interview Response Analysis Pipeline

This notebook provides a complete pipeline to analyze audio responses from mock job interviews. It transcribes spoken answers using OpenAI's Whisper model, performs sentiment classification with a zero-shot classifier, and exports structured insights into an Excel report.

## 📁 Folder Structure
The pipeline expects the following directory layout:
mock_interview_responses/
└── Candidate_Name/
└── session_id/
├── Q1.json ← Metadata for the question
├── Q1.mp3 ← Audio response (required)
├── Q2.json
├── Q2.mp3

## 🔍 What the Notebook Does
- ✅ Reads metadata from `QX.json` files.
- 🎧 Transcribes audio responses from `QX.mp3` using Whisper.
- 🧠 Classifies sentiment (e.g. *confident*, *nervous*, *neutral*) using a zero-shot transformer model.
- 📊 Outputs results into an Excel file: `interview_analysis_report.xlsx`.

## 📦 Requirements
Before running, ensure the following libraries are installed:
```bash
pip install openai-whisper transformers ffmpeg-python pandas openpyxl tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install pydub openpyxl tqdm

In [ ]:
# 📦 Install dependencies
!pip install git+https://github.com/openai/whisper.git
!pip install torch torchvision torchaudio
!pip install ffmpeg-python

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-rs7td3v7
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-rs7td3v7
  Resolved https://github.com/openai/whisper.git to commit dd985ac4b90cafeef8712f2998d62c59c3e62d22
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 134.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.6 MB/s eta 0:00:00
   ━━

In [ ]:
import os
import json
import pandas as pd
import ffmpeg
from tqdm import tqdm
from transformers import pipeline
import whisper

# === Paths ===
INTERVIEW_FOLDER = "/content/drive/MyDrive/STT SUIT/mock_interview_responses"
EXCEL_OUTPUT_PATH = "/content/interview_analysis_report.xlsx"

# === Load models ===
asr_model = whisper.load_model("large-v2")  # "small" or "base" are faster
classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")
labels = ["confident", "nervous", "neutral"]

# === Audio + Transcription + Sentiment ===
def analyze_response(audio_path, model):
    def preprocess_audio(input_path: str, output_path: str = "temp.wav") -> str:
        ffmpeg.input(input_path)\
              .output(output_path, ac=1, ar="16000", format="wav")\
              .overwrite_output()\
              .run(quiet=True)
        return output_path

    def transcribe(audio_path: str, model):
        result = model.transcribe(audio_path, verbose=False)
        transcript = " ".join([seg['text'].strip() for seg in result['segments']])
        return result, transcript

    try:
        processed_path = preprocess_audio(audio_path)
        _, transcript = transcribe(processed_path, model)
    except Exception as e:
        print(f"❌ Transcription failed for {audio_path}: {e}")
        return "", "N/A", 0.0

    if transcript.strip():
        try:
            result = classifier(transcript, candidate_labels=labels)
            sentiment = result["labels"][0]
            confidence = round(result["scores"][0], 2)
        except Exception as e:
            print(f"❌ Sentiment analysis failed: {e}")
            sentiment, confidence = "N/A", 0.0
    else:
        sentiment, confidence = "N/A", 0.0

    return transcript, sentiment, confidence

# === Main Processing ===
data = []

for candidate_name in tqdm(os.listdir(INTERVIEW_FOLDER), desc="Candidates"):
    candidate_path = os.path.join(INTERVIEW_FOLDER, candidate_name)
    if not os.path.isdir(candidate_path):
        continue

    for session_id in os.listdir(candidate_path):
        session_path = os.path.join(candidate_path, session_id)
        if not os.path.isdir(session_path):
            continue

        files = os.listdir(session_path)
        json_files = sorted([f for f in files if f.endswith(".json")])
        audio_files = sorted([f for f in files if f.endswith((".m4a", ".mp3", ".wav"))])

        for json_file in json_files:
            suffix = json_file.replace(".json", "")
            audio_match = f"{suffix}.m4a"
            json_path = os.path.join(session_path, json_file)
            audio_path = os.path.join(session_path, audio_match)

            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    meta = json.load(f)

                row = {
                    "Candidate": candidate_name,
                    "Session ID": session_id,
                    "Question": meta.get("question"),
                    "Expected Keywords": meta.get("expected_keywords"),
                    "Audio File": audio_match
                }

                if os.path.exists(audio_path):
                    transcript, sentiment, confidence = analyze_response(audio_path, asr_model)
                    row["Response Transcript"] = transcript
                    row["Delivery Sentiment"] = sentiment
                    row["Confidence Score"] = confidence
                    print(f"✅ Processed: {audio_match}")
                else:
                    row["Response Transcript"] = ""
                    row["Delivery Sentiment"] = "N/A"
                    row["Confidence Score"] = 0.0
                    print(f"⚠️ Missing audio: {audio_path}")

                data.append(row)

            except Exception as e:
                print(f"❌ Error reading {json_path}: {e}")

# === Export to Excel ===
df = pd.DataFrame(data)
df.to_excel(EXCEL_OUTPUT_PATH, index=False, engine="openpyxl")
print(f"\n✅ Interview analysis saved to: {EXCEL_OUTPUT_PATH}")


100%|█████████████████████████████████████| 2.87G/2.87G [00:47<00:00, 64.8MiB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Device set to use cuda:0
Candidates:   0%|          | 0/1 [00:00<?, ?it/s]

⚠️ Missing audio: /content/drive/MyDrive/STT SUIT/mock_interview_responses/John_Doe/session1/Q1.m4a


Candidates: 100%|██████████| 1/1 [00:01<00:00,  1.24s/it]

⚠️ Missing audio: /content/drive/MyDrive/STT SUIT/mock_interview_responses/John_Doe/session1/Q2.m4a



✅ Interview analysis saved to: /content/interview_analysis_report.xlsx


In [ ]:
import os
import json
import pandas as pd
import ffmpeg
from tqdm import tqdm
from transformers import pipeline
import whisper

# === Paths ===
INTERVIEW_FOLDER = "/content/drive/MyDrive/STT SUIT/mock_interview_responses"
EXCEL_OUTPUT_PATH = "interview_analysis_report.xlsx"

# === Load models ===
asr_model = whisper.load_model("large-v2")  # or "base", "small" if slow
classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")
labels = ["confident", "nervous", "neutral"]

# === Audio Transcription + Sentiment ===
def transcribe_and_analyze(audio_path):
    def preprocess_audio(input_path, output_path="temp.wav"):
        ffmpeg.input(input_path)\
              .output(output_path, ac=1, ar="16000", format="wav")\
              .overwrite_output()\
              .run(quiet=True)
        return output_path

    try:
        processed_path = preprocess_audio(audio_path)
        result = asr_model.transcribe(processed_path, verbose=False)
        transcript = " ".join([seg['text'].strip() for seg in result['segments']])
    except Exception as e:
        print(f"❌ Transcription failed: {e}")
        return "", "N/A", 0.0

    if transcript.strip():
        try:
            result = classifier(transcript, candidate_labels=labels)
            sentiment = result["labels"][0]
            confidence = round(result["scores"][0], 2)
        except Exception as e:
            print(f"❌ Sentiment analysis failed: {e}")
            sentiment, confidence = "N/A", 0.0
    else:
        sentiment, confidence = "N/A", 0.0

    return transcript, sentiment, confidence

# === Main Processing Loop ===
data = []

for candidate_name in tqdm(os.listdir(INTERVIEW_FOLDER), desc="Candidates"):
    candidate_path = os.path.join(INTERVIEW_FOLDER, candidate_name)
    if not os.path.isdir(candidate_path):
        continue

    for session_id in os.listdir(candidate_path):
        session_path = os.path.join(candidate_path, session_id)
        if not os.path.isdir(session_path):
            continue

        files = os.listdir(session_path)
        json_files = sorted([f for f in files if f.endswith(".json")])

        for json_file in json_files:
            qid = json_file.replace(".json", "")
            json_path = os.path.join(session_path, json_file)
            audio_path = os.path.join(session_path, f"{qid}.mp3")

            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    meta = json.load(f)

                transcript, sentiment, confidence = "", "N/A", 0.0
                if os.path.exists(audio_path):
                    transcript, sentiment, confidence = transcribe_and_analyze(audio_path)
                else:
                    print(f"⚠️ Missing audio: {audio_path}")

                row = {
                    "Candidate": candidate_name,
                    "Session ID": session_id,
                    "Question ID": qid,
                    "Question": meta.get("question"),
                    "Expected Keywords": meta.get("expected_keywords"),
                    "Transcript": transcript,
                    "Delivery Sentiment": sentiment,
                    "Confidence Score": confidence
                }

                data.append(row)
                print(f"✅ Processed: {qid}")
            except Exception as e:
                print(f"❌ Error with {json_file}: {e}")

# === Save Report ===
df = pd.DataFrame(data)
df.to_excel(EXCEL_OUTPUT_PATH, index=False, engine="openpyxl")
print(f"\n✅ Excel report saved to: {EXCEL_OUTPUT_PATH}")


100%|█████████████████████████████████████| 2.87G/2.87G [00:58<00:00, 52.6MiB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...


100%|██████████| 998/998 [00:02<00:00, 477.88frames/s]


✅ Processed: Q1


Candidates: 100%|██████████| 1/1 [00:05<00:00,  5.84s/it]

⚠️ Missing audio: /content/drive/MyDrive/STT SUIT/mock_interview_responses/John_Doe/session1/Q2.mp3
✅ Processed: Q2



✅ Excel report saved to: interview_analysis_report.xlsx
